# 决策树第七课：随机森林

本课目标：理解随机森林为什么比单棵决策树更稳定，以及它到底“随机”在哪里。

一句话版：**随机森林 = 多棵随机训练出来的决策树 + 投票。**

## 1. 为什么需要随机森林？

单棵决策树的优点是直观、容易解释，但它有一个明显问题：**容易过拟合**。

所谓过拟合，就是模型把训练集里的细枝末节也记住了。训练集上表现很好，但遇到新数据时效果变差。

例如 Titanic 案例中，如果不限制树深度，决策树可能会长得很深：

$$\text{训练集准确率很高，但验证集准确率下降}$$

随机森林的想法是：

$$\text{不要只相信一棵树，让很多棵树一起投票。}$$

## 2. 随机森林怎么做分类？

假设我们有 5 棵决策树，它们都对同一个乘客做预测：

| 树 | 预测结果 |
| ---: | --- |
| 第 1 棵树 | 生还 |
| 第 2 棵树 | 未生还 |
| 第 3 棵树 | 生还 |
| 第 4 棵树 | 生还 |
| 第 5 棵树 | 未生还 |

投票结果：

$$生还=3票,\quad 未生还=2票$$

所以最终预测：

$$\text{生还}$$

分类任务中，随机森林通常采用多数投票。

回归任务中，随机森林通常对多棵树的预测值取平均。

## 3. 随机森林为什么叫“随机”？

随机森林主要有两个随机来源：

1. 随机抽样本。
2. 随机选特征。

这两个随机的目的都是：**让每棵树长得不一样。**

如果每棵树都用完全一样的数据、完全一样的特征，那么它们很可能长得差不多。这样一群树其实和一棵树差别不大。

随机森林希望每棵树都有一点自己的视角。最后投票时，单棵树的错误会被其他树抵消。

## 4. 第一个随机：随机抽样本

随机森林训练每棵树时，不是直接使用完整训练集，而是从训练集中随机抽样。

这种抽样叫 Bootstrap sampling，也叫有放回抽样。

有放回抽样的意思是：抽到一个样本后，把它放回去，下次还有可能再次抽到它。

例如原始训练集有 8 条样本：

$$D=\{1,2,3,4,5,6,7,8\}$$

某棵树可能抽到：

$$D_1=\{1,1,3,4,4,6,7,8\}$$

另一棵树可能抽到：

$$D_2=\{2,3,3,5,6,6,7,8\}$$

注意：

- 有些样本会重复出现。
- 有些样本可能没有被抽到。
- 每棵树看到的数据都不完全一样。

## 5. 第二个随机：随机选特征

单棵决策树在每个节点分裂时，通常会从全部特征中选最优特征。

随机森林不一定每次都看全部特征，而是随机选一部分特征，再从这一部分里找最优划分。

例如 Titanic 特征有：

$$\{Pclass, Sex, Age, SibSp, Parch, Fare, Embarked\}$$

某个节点分裂时，随机森林可能只让这棵树看：

$$\{Age, Fare, Pclass\}$$

另一个节点可能只看：

$$\{Sex, SibSp, Embarked\}$$

这样可以避免所有树都过度依赖同一个强特征，也能让树之间差异更大。

## 6. 随机森林的整体流程

随机森林训练过程可以概括为：

1. 从训练集中有放回地随机抽样，得到一份子数据。
2. 用这份子数据训练一棵决策树。
3. 在每个节点分裂时，只随机考虑一部分特征。
4. 重复以上过程，训练很多棵树。
5. 预测时，让所有树投票。

写成直观形式：

```text
训练集 D
├── 抽样 D1 -> 决策树 1
├── 抽样 D2 -> 决策树 2
├── 抽样 D3 -> 决策树 3
└── 抽样 Dn -> 决策树 n

预测时：n 棵树投票
```

## 7. 为什么随机森林更稳定？

单棵树的预测可能很偏，因为它只是一棵树的判断。

随机森林让很多棵不同的树一起投票，相当于降低单棵树的偶然性。

可以这样理解：

$$\text{单棵树：方差高，容易被训练数据细节影响。}$$

$$\text{随机森林：多棵树平均后，方差降低，预测更稳。}$$

这也是集成学习的核心思想之一：

$$\text{多个弱一些或不稳定的模型组合起来，得到更稳定的整体模型。}$$

## 8. 和单棵决策树对比

| 对比点 | 单棵决策树 | 随机森林 |
| --- | --- | --- |
| 模型数量 | 1 棵树 | 很多棵树 |
| 预测方式 | 单棵树直接判断 | 多棵树投票 |
| 稳定性 | 较差，容易受数据扰动影响 | 更稳定 |
| 过拟合风险 | 较高 | 通常更低 |
| 可解释性 | 强，可以画出来 | 较弱，不容易整体解释 |
| 训练速度 | 快 | 相对慢 |
| 特征重要性 | 可以看 | 也可以看 |

## 9. 随机森林的重要参数

在 sklearn 中，常用模型是 `RandomForestClassifier`。

常见参数：

- `n_estimators`：树的数量。树越多通常越稳定，但训练更慢。
- `max_depth`：每棵树的最大深度，用来控制过拟合。
- `max_features`：每次分裂时随机考虑多少特征。
- `bootstrap`：是否使用有放回抽样，随机森林默认使用。
- `random_state`：随机种子，用来复现实验结果。
- `n_jobs`：并行训练使用的 CPU 数量。

例如：

```python
RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=22,
)
```

## 10. 袋外数据 OOB

因为随机森林训练每棵树时使用有放回抽样，所以每棵树都会有一些样本没有被抽到。

这些没有被某棵树抽到的样本，叫袋外数据，也就是 Out-Of-Bag samples，简称 OOB。

直觉上：

$$\text{某棵树没见过这些样本，所以可以用这些样本来估计这棵树的泛化表现。}$$

在 sklearn 中，可以设置：

```python
RandomForestClassifier(oob_score=True)
```

训练后可以查看：

```python
model.oob_score_
```

OOB 可以在不额外划分验证集的情况下，给出一个模型效果估计。

## 11. 随机森林的优缺点

优点：

- 通常比单棵决策树准确率更高。
- 比单棵树更不容易过拟合。
- 对特征缩放不敏感。
- 可以处理分类和回归任务。
- 可以输出特征重要性。

缺点：

- 可解释性不如单棵决策树。
- 模型更大，训练和预测更慢。
- 很难像单棵树一样完整画出来解释每一步判断。

一句话：

$$\text{随机森林牺牲了一部分可解释性，换来了更好的稳定性和泛化能力。}$$

## 12. 本课小结

- 随机森林由很多棵决策树组成。
- 每棵树使用有放回抽样得到的数据训练。
- 每个节点分裂时，随机选择一部分特征参与比较。
- 分类任务中，多棵树投票得到最终结果。
- 回归任务中，多棵树预测结果取平均。
- 随机森林通常比单棵决策树更稳定、更不容易过拟合。

记忆方式：

$$\text{随机森林 = 样本随机 + 特征随机 + 多树投票}$$